# 02: Unstructured Geodesic Mesh (UGRID NetCDF)

This notebook demonstrates 3D spherical visualization of an unstructured triangular geodesic mesh:
1. Loading node coordinates, face connectivity, and scalar fields from a UGRID-1.0 NetCDF file (`data/unstructured/unstructured_geodesic_mesh.nc`).
2. Constructing an unstructured spherical 3D mesh with `geovista.Transform.from_unstructured`.
3. Highlighting cell wireframe edges (`show_edges=True`, `edge_color="navy"`).
4. Visualizing physical oceanographic/atmospheric variables: **vorticity** and **sea surface height (SSH)**.

In [ ]:
import os
from pathlib import Path
import netCDF4 as nc
import numpy as np
import geovista as gv
from IPython.display import Image, display

import pyearthviz3d
from pyearthviz3d import (
    VisualizationConfig,
    ScalarBarConfig,
    map_single_frame,
)

# Resolve repository root and data path
REPO_ROOT = Path(os.getcwd()).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

UGRID_PATH = REPO_ROOT / "data" / "unstructured" / "unstructured_geodesic_mesh.nc"
print(f"pyearthviz3d version: {pyearthviz3d.__version__}")
print(f"Target UGRID dataset: {UGRID_PATH} (exists: {UGRID_PATH.exists()})")

## Step 1: Load and Inspect the UGRID NetCDF Dataset

Unstructured climate and ocean models (such as MPAS, ICON, and SE-HOMME) store mesh coordinates as 1D arrays of nodes along with a 2D connectivity table specifying node indices for each triangular or polygonal face.

In [ ]:
with nc.Dataset(UGRID_PATH, "r") as ds:
    node_lons = ds.variables["Mesh2_node_x"][:]
    node_lats = ds.variables["Mesh2_node_y"][:]
    face_nodes = ds.variables["Mesh2_face_nodes"][:]
    vorticity = ds.variables["vorticity"][:]
    ssh = ds.variables["sea_surface_height"][:]

print(f"Number of nodes: {len(node_lons)}")
print(f"Number of triangular faces: {len(face_nodes)}")
print(f"Face connectivity shape: {face_nodes.shape} (3 nodes per face)")
print(f"Vorticity range: [{vorticity.min():.2e}, {vorticity.max():.2e}] 1/s")
print(f"SSH range:       [{ssh.min():.2f}, {ssh.max():.2f}] m")

## Step 2: Build Unstructured 3D Spherical Mesh

Using `geovista.Transform.from_unstructured`, we assemble the spherical polyhedron by projecting nodes onto the sphere and connecting triangular cells according to `face_nodes`.

In [ ]:
mesh = gv.Transform.from_unstructured(
    xs=node_lons,
    ys=node_lats,
    connectivity=face_nodes,
    data=vorticity,
    name="vorticity",
    start_index=0,
)
mesh.cell_data["sea_surface_height"] = ssh
valid_indices = np.arange(mesh.n_cells)

print(f"Constructed PyVista PolyData: {mesh.n_cells} cells, {mesh.n_points} vertices.")

## Step 3: Visualize Vorticity with Wireframe Mesh Edges

We center the globe over the North Atlantic (`longitude_focus=-30.0`, `latitude_focus=40.0`), display individual triangular cell boundaries with `show_edges=True`, and apply a diverging `coolwarm` colormap.

In [ ]:
viz_config = VisualizationConfig(
    longitude_focus=-30.0,  # North Atlantic vantage
    latitude_focus=40.0,
    zoom_factor=0.85,
    show_coastlines=True,
    show_graticule=True,
    coastline_color="black",
    coastline_width=1.5,
    colormap="coolwarm",
    verbose=False,
)

scalar_config = ScalarBarConfig(
    title="Relative Vorticity",
    title_font_size=12,
    label_font_size=10,
    fmt="%.1e",
    n_labels=5,
    orientation="horizontal",
)

output_png = "unstructured_mesh_vorticity.png"
result = map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    sScalar="vorticity",
    sUnit="1/s",
    sFilename_out=output_png,
    scalar_config=scalar_config,
    show_edges=True,
    edge_color="navy",
    return_detailed_result=True,
)

print(result.get_summary())
display(Image(output_png, width=650))

## Step 4: Visualize Sea Surface Height (SSH)

Now we render the dynamic sea surface height anomaly using the `viridis` sequential colormap.

In [ ]:
viz_ssh_config = VisualizationConfig(
    longitude_focus=150.0,  # Pacific Ocean focus
    latitude_focus=10.0,
    zoom_factor=0.85,
    show_coastlines=True,
    show_graticule=True,
    coastline_color="white",
    coastline_width=1.2,
    colormap="viridis",
    verbose=False,
)

scalar_ssh_config = ScalarBarConfig(
    title="Sea Surface Height Anomaly",
    title_font_size=12,
    label_font_size=10,
    fmt="%.2f",
    n_labels=6,
    orientation="horizontal",
)

output_ssh_png = "unstructured_mesh_ssh.png"
map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_ssh_config,
    sScalar="sea_surface_height",
    sUnit="m",
    sFilename_out=output_ssh_png,
    scalar_config=scalar_ssh_config,
    show_edges=True,
    edge_color="gray",
)

display(Image(output_ssh_png, width=650))